<H1>TL_Example</H1> 
Transmission line example, Simple case or first one on the page. 
power lost due to a lossy line into a resistive load. 
<hr>
<H2>Modification history </H2><br> 
<table>
    <tr>
        <th> Modified </th>
        <th> By </th>
        <th> Reason </th>
    </tr>
    <tr> 
        <td> 07-Mar-26 </td>
        <td> CBL </td> 
        <td> Original </td>
    </tr>
</table>
<hr> 
<H2> References </H2> 
<a href = "https://scikit-rf.readthedocs.io/en/v1.8.0/examples/networktheory/Transmission%20Line%20Losses.html"> Transmission Line Example </a><br> 
<H2> Notes </H2>

In [4]:
# imports - package libraries etc. 
import numpy as np
import matplotlib.pyplot as plt
import skrf as rf
from skrf import Network
from skrf import Frequency
from skrf.circuit import Circuit
from skrf.tlineFunctions import INF
from skrf.mathFunctions import db_2_np
#
# skrf figure styling
rf.stylely()

Define the constants of the problem. The Load is the same as the transmission line. 

In [2]:
Pin = 400  # W
z0 = 50 # Ohm
freq = rf.Frequency(13.56, npoints=1, unit='MHz')
VF = 0.84
RL = 50  # Ohm
L = 20  # m

The propigation constant of the transmission line is: $\gamma = \alpha + j\beta$

In [6]:
alpha = db_2_np(1.483/100)  # Np/m
beta = freq.w/rf.constants.c/VF
gamma = alpha + 1j*beta

The matched line losss (or power attenuation ), $a=e^{2aL}$

In [8]:
a = np.exp(2*alpha*L)  # also simply 2.84/100*20
print('Matched line loss: a=', rf.mathFunctions.mag_2_db10(a), 'dB')

Matched line loss: a= 0.29660000000000003 dB


If the line is charged with a matched load, ie. , then the total line loss is . The power lost in the cable will thus be:

In [9]:
print('(Forward) Power delivered to the  load:', Pin/a, 'W')
print('Power lost in the cable:', Pin *( 1 - 1/a), 'W')

(Forward) Power delivered to the  load: 373.59408479883416 W
Power lost in the cable: 26.405915201165886 W


which can also be checked by the scikit-rf transmission line function

In [11]:
a_skrf = rf.tlineFunctions.zl_2_total_loss(z0, zl=RL, theta=gamma*L)
print('Power lost in the cable:',
      Pin * (1 - 1/a_skrf), 'W')

Power lost in the cable: [26.4059152] W


Another way to evaluate the toal power dissipated in the circuit is to evaluate the power expression:
$P_{delivered} = {1\over 2} \mathbb{R} [VI^*]$
Where V and I are the (peak) total voltage and current. They can be evalued using the transmission line function 

In [14]:
# reflection coefficient and input impedance
Gamma_in = rf.tlineFunctions.zl_2_Gamma_in(z0, RL, theta=gamma*L)
Z_in = rf.tlineFunctions.zl_2_zin(z0, RL, theta=gamma*L)

# voltage and current at the line input as a function of source power
V_in = np.sqrt(2*z0*Pin)*(1 + Gamma_in)
I_in = V_in/Z_in

# voltage and current at z=L
V,I = rf.tlineFunctions.voltage_current_propagation(V_in, I_in, z0, gamma*L)
P_delivered = 1/2 * np.real(V * np.conj(I))
print('Power delivered to the load: ', P_delivered, 'W')
print('Power dissipated in the cable: ',Pin - P_delivered, 'W')

Power delivered to the load:  [373.5940848] W
Power dissipated in the cable:  [26.4059152] W


<H1> Non Matched Load</H1>
However, if the load is not perfectly matched to the line characteristic impedance $z_0$, for exxample with $R_L = 200 + 30j\Omega$, additional losses are induced by the reflected wave. The reflection coefficient $\Gamma_{load}$

In [15]:
z0 = 50
ZL = 200 - 30j
Gamma_load = rf.tlineFunctions.zl_2_Gamma0(z0, ZL)
print('|Gamma_load|=', np.abs(Gamma_load))

|Gamma_load|= [0.6075238]


while the reflection coefficient seen at the input of the transmission line $\Gamma_{in}$ is

In [17]:
Gamma_in = rf.tlineFunctions.zl_2_Gamma_in(z0, ZL, theta=gamma*L)
SWR = rf.tlineFunctions.Gamma0_2_swr(rf.tlineFunctions.zl_2_Gamma_in(z0, ZL, theta=gamma*L))
print('|Gamma_in|=', np.abs(Gamma_in), '(SWR=', SWR,')')

|Gamma_in|= [0.56741825] (SWR= [3.62340355] )


The total loss in dB due to SWR is often stated as:
$a_{[dB} + 10 log_{10} {{1-|\Gamma_{in}|^2}\over{1-|\Gamma_{load}|^2}}$

In [18]:
10*np.log10(a) + 10*np.log10((1 - np.abs(Gamma_in)**2)/(1 - np.abs(Gamma_load)**2))

array([0.60942359])

However, this expression is only correct if either properties are verified
<ol>
    <li> the characteristic impedance of the line is real (distorsionless line)</li>
    <li> reflection coefficient is real (ie real ${Z_L}\over{Z_0}$) </li> 
</ol>
The first condition is met here, however it will not be the case in the next section. The scikit-rf transmission line function is correct for all conditions. 

In [20]:
a = rf.tlineFunctions.zl_2_total_loss(z0, zl=ZL, theta=gamma * L)
print('Total power loss: ', rf.mathFunctions.mag_2_db10(a), 'dB' )
print('Delivered power:', Pin/a, 'W')
print('The total power loss is the cable:', Pin*(1 - 1/a), 'W')

Total power loss:  [0.60942359] dB
Delivered power: [347.63030721] W
The total power loss is the cable: [52.36969279] W


In [22]:
# reflection coefficient and input impedance
Gamma_in = rf.tlineFunctions.zl_2_Gamma_in(z0, ZL, theta=gamma*L)
Z_in = rf.tlineFunctions.zl_2_zin(z0, ZL, theta=gamma*L)

# voltage and current at the line input as a function of source power
V_in = np.sqrt(2*z0*Pin)*(1 + Gamma_in)
I_in = V_in/Z_in

# voltage and current at z=L
V,I = rf.tlineFunctions.voltage_current_propagation(V_in, I_in, z0, gamma*L)
P_delivered = 1/2 * np.real(V * np.conj(I))
print('Power delivered to the load: ', P_delivered, 'W')
print('Power dissipated in the cable: ',Pin - P_delivered, 'W')

Power delivered to the load:  [235.70604719] W
Power dissipated in the cable:  [164.29395281] W


In [23]:
rf.tlineFunctions.Gamma0_2_swr(Gamma_in)

array([3.62340355])

In [24]:
10*np.log10(P_delivered/Pin)

array([-2.29689267])